# E2B 代码沙箱 · 完全入门教程

**一句话定位**：E2B 为 AI agent 按需提供一个隔离、安全、可联网的云端 Linux microVM，让大模型生成的代码有一个能真正"跑起来"且不会伤害宿主的地方。

本教程按"**原理 → 推论 → 接口 → 实操**"组织，覆盖从第一个沙箱到把沙箱接入 LLM 的完整链路：

| 章节 | 主题 | 关键能力 |
|---|---|---|
| 一 | 为什么需要沙箱 | 理解 E2B 解决的根本问题 |
| 二 | 架构与心智模型 | microVM、两个 SDK 包、生命周期地图 |
| 三 | 环境准备 | 安装、`E2B_API_KEY` |
| 四~六 | 执行代码 | `run_code`、`Execution` 对象、有状态 context、错误处理 |
| 七 | 文件系统 | `sbx.files` 读写/上传下载/监听 |
| 八 | 命令执行 | `sbx.commands` 同步/流式/后台 |
| 九 | 生命周期 | 超时、`pause`/`connect`、`list` |
| 十 | 数据分析与图表 | 富输出 `results` |
| 十一 | 接入 LLM | Claude tool-use 闭环 |
| 十二~十四 | 自定义模板 / 生产实践 / 速查表 | 落地工程化 |

> **运行说明**：本 notebook 的代码单元可直接运行，前提是已安装 `e2b-code-interpreter` 并设置了 `E2B_API_KEY`（见第三章）。建议在 VSCode 或 Jupyter 中打开。每个沙箱都会消耗配额，运行完记得让其关闭（用 `with` 语句或 `kill`）。

## 一、为什么需要沙箱（原理）

大模型能写代码，但写出来的代码必须在某个地方**执行**才能产生价值——做数学计算、跑数据分析、调用工具、运行测试。问题在于：这段代码来自模型，不可信、不可预测，可能死循环、可能 `rm -rf`、可能读取宿主机密钥。

把它直接放在自己的服务器进程里执行，等于把方向盘交给一个不认识的人。于是需要一个**隔离的、一次性的执行环境**：

- **隔离**：代码看不见也碰不到宿主机的文件、网络、其他用户的数据。
- **一次性**：用完即弃，环境被污染了也无所谓，下次重新开一个干净的。
- **按需**：agent 随时可能需要执行代码，环境必须能在一两秒内起好。
- **完整**：它得是一台真正的 Linux 机器——能装包、能联网、能跑任意进程，而不只是一个受限的 `eval`。

这正是 **E2B Sandbox** 的定位：一台**按需创建、秒级启动、用完即焚的安全云端 Linux 虚拟机**。

**由这个定位可以推出它必然要提供的几类能力**（也就是本教程的主干）：

1. **执行代码** —— 把一段代码丢进去运行并取回结果（`run_code`）。
2. **读写文件** —— 把数据送进去、把产物取出来（`files`）。
3. **运行命令** —— 像在终端里一样跑任意 shell 命令、装依赖（`commands`）。
4. **管理生命周期** —— 控制存活时长、暂停与恢复、销毁（`timeout` / `pause` / `kill`）。
5. **定制环境** —— 预装依赖做成模板，加速冷启动（Template）。
6. **接入大模型** —— 让 LLM 把"执行代码"当成一个工具来调用。

## 二、整体架构与心智模型

### 2.1 沙箱是什么：一台 microVM

E2B 的每个 Sandbox 都是一台基于 **Firecracker microVM** 的轻量虚拟机——不是容器，而是有独立内核的虚拟机，隔离强度更高，启动却只要约 150ms。它内部预装了 Python、Node.js、常用数据科学库，并跑着一个叫 **envd** 的守护进程，SDK 的所有操作（执行代码、读写文件、跑命令）本质上都是在和这个守护进程通信。

```mermaid
flowchart LR
    A[本地程序 / Agent] -->|SDK 调用| B[E2B 控制面]
    B -->|按需启动| C[云端 microVM 沙箱]
    subgraph C[云端 microVM 沙箱]
        D[守护进程 envd]
        E[Jupyter 内核]
        F[文件系统]
        G[Shell 与进程]
    end
    D --- E
    D --- F
    D --- G
```

### 2.2 两个 SDK 包：核心包 vs 代码解释器包

E2B 有两个层次的 Python 包，**入门请直接用上层的 `e2b-code-interpreter`**（它包含了核心包的全部能力）：

| 包 | 导入 | 提供的东西 | 何时用 |
|---|---|---|---|
| `e2b`（核心） | `from e2b import Sandbox` | 沙箱生命周期、`files`、`commands`、`pty` | 只需跑 shell 命令、传文件，不需要"有状态地执行代码" |
| `e2b-code-interpreter`（上层） | `from e2b_code_interpreter import Sandbox` | 上面全部 **＋ `run_code`**：沙箱内置一个 **有状态的 Jupyter 内核**，能记住变量、捕获图表等富输出 | 需要让模型反复执行代码、做数据分析、画图——**绝大多数场景** |

> 关键区别：核心包能"跑命令"，上层包额外能"像在 Jupyter 里一样跑代码并保留状态"。本教程统一使用 `e2b-code-interpreter`。

### 2.3 生命周期地图

一个沙箱从生到死的状态流转如下，后面第九章会逐一展开：

```mermaid
flowchart LR
    A[发起创建] --> B[云端启动 microVM]
    B --> C[运行中]
    C -->|执行代码 / 跑命令 / 读写文件| C
    C -->|主动关闭| E[销毁]
    C -->|空闲到达超时| D{超时策略}
    D -->|默认: 关闭| E
    D -->|可选: 暂停| F[已暂停<br/>状态被冻结保存]
    F -->|重新连接| C
```

## 三、环境准备

**第一步**：去 [e2b.dev/dashboard](https://e2b.dev/dashboard) 注册并拿到 API key（形如 `e2b_xxx`）。

**第二步**：安装包（顺带装 `python-dotenv` 方便管理 key）。

**第三步**：把 key 放进环境变量。最简单是在项目根目录建一个 `.env` 文件：

```
E2B_API_KEY=e2b_***
```

下面两个单元分别完成"安装"和"加载并校验 key"。

In [ ]:
# 安装（已装可跳过）。e2b-code-interpreter 会自动带上核心 e2b 包。
%pip install -q e2b-code-interpreter python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # 从当前目录的 .env 读取 E2B_API_KEY 到环境变量

assert os.environ.get("E2B_API_KEY"), "未检测到 E2B_API_KEY，请在 .env 或环境变量中设置"
print("E2B_API_KEY 已就绪：", os.environ["E2B_API_KEY"][:8] + "…")

## 四、第一个 Sandbox

**原理**：`Sandbox.create()` 向 E2B 控制面请求，在云端启动一台 microVM 并返回一个本地句柄；之后所有 `sbx.xxx` 调用都通过这个句柄远程操作那台机器。沙箱有存活时长上限（`timeout`，**单位秒**，默认 300 秒 / 5 分钟），到点自动回收，避免忘记关闭而持续计费。

**核心签名**：

```python
Sandbox.create(
    template: str | None = None,        # 用哪个环境模板，默认是官方基础模板
    timeout: int | None = 300,          # 存活秒数，默认 300s
    metadata: dict[str, str] | None = None,   # 自定义标签，便于之后检索
    envs: dict[str, str] | None = None,       # 注入到沙箱里的环境变量
    secure: bool = ...,                 # 是否启用受保护访问
    allow_internet_access: bool = True, # 沙箱是否可以出网
) -> Sandbox
```

**最佳实践**：用 `with` 语句创建，离开代码块时自动 `kill`，绝不泄漏沙箱。

In [ ]:
from e2b_code_interpreter import Sandbox

# 用 with 确保用完即关。timeout=120 表示这台沙箱最多存活 120 秒。
with Sandbox.create(timeout=120) as sbx:
    print("沙箱已创建，ID =", sbx.sandbox_id)

    execution = sbx.run_code("print('hello world')")
    print("输出：", execution.logs.stdout)

    # 顺便看看沙箱里的根目录有哪些东西
    entries = sbx.files.list("/")
    print("根目录条目数：", len(entries))
# 离开 with 后，沙箱已被自动销毁

## 五、`run_code` 与 `Execution` 对象解剖

**原理**：`run_code` 把代码送进沙箱内置的 **Jupyter 内核**执行。既然是 Jupyter 内核，就有两个重要推论：

1. **有状态**：同一个沙箱里，前一次定义的变量、导入的库，后一次还能用——就像在同一个 notebook 里跑多个 cell。
2. **富输出**：内核不仅返回打印的文本，还能捕获图片、图表、HTML、Markdown 等"最后一个表达式的渲染结果"——这正是数据分析场景的关键。

**签名**（常用参数）：

```python
sbx.run_code(
    code: str,
    language: str | None = None,   # 'python'（默认）/ 'js' / 'r' / 'bash' / 'java'
    context = None,                # 指定一个代码上下文，见 5.3
    on_stdout = None,              # 流式回调
    on_stderr = None,
    timeout: float | None = None,
) -> Execution
```

**`Execution` 对象的结构**（务必记住这张图）：

| 字段 | 类型 | 含义 |
|---|---|---|
| `execution.logs.stdout` | `list[str]` | 标准输出（`print` 的内容） |
| `execution.logs.stderr` | `list[str]` | 标准错误 |
| `execution.results` | `list[Result]` | 富输出列表，每个 `Result` 有 `.text` / `.png` / `.jpeg` / `.html` / `.markdown` / `.svg` / `.chart` 等属性 |
| `execution.error` | `ExecutionError \| None` | 运行报错时的 `.name` / `.value` / `.traceback`，没报错则为 `None` |
| `execution.text` | `str \| None` | 快捷方式：最后一个结果的文本表示 |

In [ ]:
with Sandbox.create(timeout=120) as sbx:
    execution = sbx.run_code(
        """
import math
print("这行进入 stdout")
result = math.sqrt(144)
result  # 最后一个表达式 → 进入 results
"""
    )

    print("stdout      :", execution.logs.stdout)
    print("stderr      :", execution.logs.stderr)
    print("results     :", execution.results)  # [Result(...)]
    print("results[0].text:", execution.results[0].text)  # '12.0'
    print("execution.text :", execution.text)  # 同上的快捷方式
    print("error       :", execution.error)  # None

### 5.2 有状态性：跨多次执行共享变量

下面证明"同一个沙箱里变量会被记住"——这是 `run_code` 区别于普通无状态执行的核心。

In [ ]:
with Sandbox.create(timeout=120) as sbx:
    sbx.run_code("x = 41")  # 第一次执行：定义变量
    e = sbx.run_code("x + 1")  # 第二次执行：仍能访问 x
    print("x + 1 =", e.text)  # 42 —— 状态被内核保留

### 5.3 代码上下文（context）与多语言

**原理**：默认情况下同一沙箱共享一个内核。若想要**多个互相隔离的状态空间**（比如指定不同工作目录或语言），可显式创建 `context`，再把它传给 `run_code`。

**签名**：

```python
ctx = sbx.create_code_context(
    cwd: str | None = None,        # 该上下文的工作目录
    language: str | None = None,   # 'python' / 'js' / 'r' / ...
)
sbx.run_code(code, context=ctx)    # 在指定上下文里执行
```

也可以不建 context，直接用 `language=` 跑一段其它语言的代码（一次性，不保留该语言的状态）。

In [ ]:
with Sandbox.create(timeout=120) as sbx:
    # 方式 A：显式上下文，绑定工作目录
    ctx = sbx.create_code_context(cwd="/home/user", language="python")
    print(sbx.run_code("import os; os.getcwd()", context=ctx).text)  # /home/user

    # 方式 B：直接指定语言跑一段 JavaScript
    js = sbx.run_code(
        "console.log('hello from node ' + process.version)", language="js"
    )
    print(js.logs.stdout)

## 六、错误处理

**关键认知**：沙箱里代码抛异常时，`run_code` **不会**在本地抛出异常，而是把错误信息塞进 `execution.error`。因此判断"这次执行成没成功"要看 `execution.error` 是否为 `None`，而不是用 `try/except` 包住 `run_code`。

`ExecutionError` 三个字段：`.name`（异常类名）、`.value`（异常消息）、`.traceback`（完整堆栈字符串）。

In [ ]:
with Sandbox.create(timeout=120) as sbx:
    execution = sbx.run_code("1 / 0")

    if execution.error:
        print("捕获到沙箱内异常：")
        print("  name :", execution.error.name)  # ZeroDivisionError
        print("  value:", execution.error.value)  # division by zero
        # execution.error.traceback 里是完整堆栈，可回灌给 LLM 让它自我修正
    else:
        print("结果：", execution.text)

## 七、文件系统 `sbx.files`

**原理**：沙箱有自己独立的文件系统，默认家目录是 `/home/user`。所有文件操作通过 `sbx.files` 这个命名空间完成，是把数据"喂进去"和把产物"取出来"的通道。

**完整签名**：

| 方法 | 签名 | 说明 |
|---|---|---|
| 读 | `read(path, format="text"\|"bytes"\|"stream")` | 返回 `str` / `bytearray` / 字节流迭代器 |
| 写 | `write(path, data)` | `data` 可为 `str` / `bytes` / 文件对象 |
| 批量写 | `write([{ "path":..., "data":... }, ...])` | 一次写多个文件 |
| 列目录 | `list(path, depth=None)` | 返回 `list[EntryInfo]` |
| 是否存在 | `exists(path) -> bool` | |
| 删除 | `remove(path)` | |
| 重命名/移动 | `rename(old_path, new_path)` | |
| 建目录 | `make_dir(path) -> bool` | |
| 监听变更 | `watch_dir(path, recursive=False)` | 返回 `WatchHandle` |

In [ ]:
with Sandbox.create(timeout=120) as sbx:
    # 写入与读取
    sbx.files.write("/home/user/hello.txt", "你好，沙箱")
    print("读回：", sbx.files.read("/home/user/hello.txt"))

    # 批量写多个文件
    sbx.files.write(
        [
            {"path": "/home/user/a.txt", "data": "AAA"},
            {"path": "/home/user/b.txt", "data": "BBB"},
        ]
    )

    # 建目录、列目录、判存在、重命名、删除
    sbx.files.make_dir("/home/user/data")
    sbx.files.rename("/home/user/a.txt", "/home/user/data/a.txt")
    print("data 目录：", [e.name for e in sbx.files.list("/home/user/data")])
    print("b.txt 存在？", sbx.files.exists("/home/user/b.txt"))
    sbx.files.remove("/home/user/b.txt")
    print("删除后还在吗？", sbx.files.exists("/home/user/b.txt"))

### 7.2 上传本地文件 / 下载沙箱产物

**原理**：上传 = 把本地读出的字节 `write` 进沙箱；下载 = 用 `read(..., format="bytes")` 取出字节再落盘。文本直接传 `str` 即可，二进制（图片、模型、压缩包）必须走 `bytes`。

In [ ]:
with Sandbox.create(timeout=120) as sbx:
    # —— 上传：本地文件 → 沙箱 ——
    # with open("local_input.csv", "rb") as f:
    #     sbx.files.write("/home/user/input.csv", f.read())

    # 这里直接在沙箱里造一个二进制产物来演示下载
    sbx.run_code(
        """
with open("/home/user/report.bin", "wb") as f:
    f.write(bytes(range(256)))
"""
    )

    # —— 下载：沙箱 → 本地 ——
    data = sbx.files.read("/home/user/report.bin", format="bytes")
    with open("/tmp/report_downloaded.bin", "wb") as f:
        f.write(data)
    print("已下载字节数：", len(data))

### 7.3 监听目录变更 `watch_dir`

当沙箱里有长时间运行的任务在不断产出文件时，可以监听目录，实时拿到"新增 / 修改 / 删除"事件，而不必轮询。

In [ ]:
with Sandbox.create(timeout=120) as sbx:
    sbx.files.make_dir("/home/user/watched")
    handle = sbx.files.watch_dir("/home/user/watched")

    # 在被监听目录里造一些动静
    sbx.files.write("/home/user/watched/new.txt", "hi")

    for event in handle.get_new_events():
        print("事件：", event.type, event.name)

## 八、命令执行 `sbx.commands`

**`run_code` vs `commands.run` 的分工**：

- `run_code`：在 **Jupyter 内核**里执行代码，**有状态**，能取回富输出——适合"计算、分析、画图"。
- `commands.run`：在 **shell** 里执行任意命令，**无内核状态**——适合"装依赖、跑脚本、起服务、用命令行工具"，比如 `pip install`、`git clone`、`node server.js`。

**签名**：

```python
sbx.commands.run(
    cmd: str,
    background: bool = False,        # True 则立即返回句柄，命令在后台跑
    cwd: str | None = None,          # 工作目录
    envs: dict[str, str] | None = None,
    on_stdout = None,                # 实时输出回调
    on_stderr = None,
    timeout: float | None = None,
) -> CommandResult | CommandHandle
```

同步执行返回 `CommandResult`，含 `.stdout` / `.stderr` / `.exit_code`。

In [ ]:
with Sandbox.create(timeout=120) as sbx:
    result = sbx.commands.run("echo hello && uname -a")
    print("exit_code:", result.exit_code)
    print("stdout   :", result.stdout)

    # 典型用途：在沙箱里临时装一个包
    install = sbx.commands.run("pip install -q cowsay")
    print("安装退出码：", install.exit_code)

### 8.2 流式输出：边跑边收

长命令不想等它跑完再一次性拿输出时，传 `on_stdout` / `on_stderr` 回调，输出会一行行实时推过来。

In [ ]:
with Sandbox.create(timeout=120) as sbx:
    sbx.commands.run(
        "for i in 1 2 3; do echo line-$i; sleep 1; done",
        on_stdout=lambda line: print("实时:", line, end=""),
    )

### 8.3 后台命令：起一个长期进程

`background=True` 会立即返回一个 `CommandHandle`，命令在沙箱里持续运行（比如起一个 web 服务）。句柄可迭代（产出 `(stdout, stderr, _)`），也有 `.wait()`、`.kill()`、`.pid`。

In [ ]:
with Sandbox.create(timeout=120) as sbx:
    cmd = sbx.commands.run("echo start; sleep 30; echo done", background=True)
    print("后台进程 PID：", cmd.pid)

    # 读几行输出后主动结束它（演示用；真实场景常让它一直跑）
    for stdout, stderr, _ in cmd:
        if stdout:
            print("后台输出:", stdout.strip())
            break
    cmd.kill()
    print("后台命令已终止")

## 九、生命周期管理

### 9.1 超时与基本操作

沙箱默认存活 300 秒，到点自动销毁。可在创建时设 `timeout`，运行中用 `set_timeout` 续命（**新超时从调用那一刻重新计时**）。

| 操作 | 实例方法 | 静态方法（凭 ID 远程操作） |
|---|---|---|
| 续命 | `sbx.set_timeout(秒)` | `Sandbox.set_timeout(sandbox_id, 秒)` |
| 查信息 | `sbx.get_info()` | `Sandbox.get_info(sandbox_id)` |
| 销毁 | `sbx.kill()` | `Sandbox.kill(sandbox_id)` |

`get_info()` 返回的信息含 `sandbox_id`、`template_id`、`name`、`metadata`、`started_at`、`end_at` 等。

In [ ]:
sbx = Sandbox.create(timeout=60, metadata={"用途": "教程演示", "env": "dev"})
try:
    info = sbx.get_info()
    print("ID      :", info.sandbox_id)
    print("metadata:", info.metadata)

    sbx.set_timeout(120)  # 从现在起再续到 120 秒
    print("已续命到 120 秒")
finally:
    sbx.kill()  # 不用 with 时，务必手动销毁
    print("已销毁")

### 9.2 列出沙箱 `Sandbox.list`

`Sandbox.list()` 返回一个**分页器**，可按状态、metadata 过滤。常用于运维：查现在有哪些沙箱在跑、清理僵尸沙箱。

In [ ]:
from e2b import SandboxQuery, SandboxState

paginator = Sandbox.list(
    query=SandboxQuery(state=[SandboxState.RUNNING]),  # 只看运行中的
)
running = paginator.next_items()
while paginator.has_next:
    running.extend(paginator.next_items())

print("当前运行中的沙箱数：", len(running))
for s in running[:5]:
    print(" ", s.sandbox_id, s.state, s.metadata)

### 9.3 暂停与恢复（persistence）

**原理**：`pause()` 会把沙箱的**完整状态——文件系统、内存、运行中的进程、内核里的变量——全部冻结保存**，沙箱停止计费。之后用 `connect(sandbox_id)` 重新连上时会自动恢复，仿佛从没停过。这让"长期会话""跨请求复用同一环境"成为可能。

**要点**：
- `sbx.pause()` 暂停；返回的 `sandbox_id` 用于以后恢复。
- `Sandbox.connect(sandbox_id)` 连接并自动恢复（没有单独的 `resume` 方法）。
- 暂停耗时约 **4 秒 / GiB 内存**；暂停态可无限期保存，不会自动删除。
- 也可让沙箱"超时即暂停"而非销毁：`lifecycle={"on_timeout": "pause"}`。

```mermaid
flowchart LR
    A[运行中] -->|pause| B[已暂停<br/>文件+内存+进程被冻结]
    B -->|connect 沙箱ID| C[自动恢复<br/>状态完整还原]
    C --> A
```

In [ ]:
# 暂停一个沙箱，记下它的 ID
sbx = Sandbox.create(timeout=120)
sbx.run_code("secret = 12345")  # 在内存里留个变量
sid = sbx.sandbox_id
sbx.pause()
print("已暂停：", sid)

# —— 之后（甚至在另一个进程 / 另一台机器上）—— 凭 ID 恢复
restored = Sandbox.connect(sid)
print("恢复后变量还在吗：", restored.run_code("secret").text)  # 12345
restored.kill()

## 十、数据分析与图表（招牌能力）

**原理**：沙箱模板预装了 `pandas`、`numpy`、`matplotlib` 等数据科学库。当代码画了一张图，Jupyter 内核会把它捕获成富输出，放进 `execution.results[0]`，其中 `.png` 是 base64 编码的图片数据。这意味着 **agent 不仅能算出数字，还能产出可视化图表**，直接回传给前端展示。

In [ ]:
import base64

plot_code = """
import matplotlib.pyplot as plt
import numpy as np

x = np.linspace(0, 2 * np.pi, 200)
plt.figure(figsize=(6, 3))
plt.plot(x, np.sin(x), label="sin")
plt.plot(x, np.cos(x), label="cos")
plt.legend(); plt.title("sin & cos")
plt.show()
"""

with Sandbox.create(timeout=120) as sbx:
    execution = sbx.run_code(plot_code)
    result = execution.results[0]

    if result.png:  # base64 字符串
        with open("/tmp/e2b_chart.png", "wb") as f:
            f.write(base64.b64decode(result.png))
        print("图表已保存到 /tmp/e2b_chart.png")
    print(
        "该结果可用的富输出类型：",
        [
            k
            for k in ("png", "jpeg", "svg", "html", "chart")
            if getattr(result, k, None)
        ],
    )

## 十一、把 E2B 接到 LLM（核心场景）

**原理**：大模型本身不能执行代码。要让它"会算数、会跑分析"，标准做法是给它一个名为 `execute_python` 的**工具（tool）**，工具的实现就是 `sbx.run_code`。于是形成一个闭环：

```mermaid
flowchart LR
    U[用户提问] --> M[大模型]
    M -->|决定要算/查| T[请求执行一段代码]
    T --> S[沙箱执行并返回结果]
    S -->|输出回灌| M
    M -->|拿到结果后作答| A[最终回答]
```

模型生成代码 → 程序把代码丢进沙箱执行 → 把执行结果（含错误堆栈）回灌给模型 → 模型据此继续推理或修正，直到给出最终答案。下面用 Anthropic Claude 演示这个闭环。

> 需要额外设置 `ANTHROPIC_API_KEY`，并 `pip install anthropic`。模型 ID 用 `claude-sonnet-4-6`（可按需替换为其它型号）。

In [ ]:
%pip install -q anthropic

In [ ]:
from anthropic import Anthropic
from e2b_code_interpreter import Sandbox

client = Anthropic()
MODEL = "claude-sonnet-4-6"

tools = [
    {
        "name": "execute_python",
        "description": "在一个有状态的 Jupyter 沙箱里执行 Python 代码，返回 stdout 与结果。需要计算、数据处理或验证时使用。",
        "input_schema": {
            "type": "object",
            "properties": {
                "code": {"type": "string", "description": "要执行的 Python 代码"}
            },
            "required": ["code"],
        },
    }
]


def run_agent(question: str) -> str:
    """运行一轮带代码执行能力的 agent 循环，返回模型的最终文本回答。

    参数:
        question: 用户的问题。
    返回:
        模型在多轮工具调用后给出的最终自然语言回答。
    """
    messages = [{"role": "user", "content": question}]

    with Sandbox.create(timeout=180) as sbx:  # 整个会话共用一个沙箱，状态连续
        while True:
            resp = client.messages.create(
                model=MODEL, max_tokens=1024, messages=messages, tools=tools
            )
            messages.append({"role": "assistant", "content": resp.content})

            if resp.stop_reason != "tool_use":  # 模型不再要工具 → 已是最终答案
                return "".join(b.text for b in resp.content if b.type == "text")

            tool_results = []
            for block in resp.content:
                if block.type == "tool_use" and block.name == "execute_python":
                    execution = sbx.run_code(block.input["code"])
                    # 把输出或错误堆栈回灌给模型
                    payload = (
                        execution.error.traceback
                        if execution.error
                        else (execution.text or "\n".join(execution.logs.stdout))
                    )
                    tool_results.append(
                        {
                            "type": "tool_result",
                            "tool_use_id": block.id,
                            "content": payload or "(无输出)",
                        }
                    )
            messages.append({"role": "user", "content": tool_results})


print(run_agent("单词 'strawberry' 里有几个字母 r？用代码数一遍来确认。"))

## 十二、自定义环境模板（Template）

**为什么需要**：默认模板每次冷启动后，agent 若要用某个未预装的库，得现 `pip install`，既慢又重复。把依赖**预先打进模板**，沙箱一启动就带着这些库，冷启动快、行为可复现。

**两种方式**：

1. **CLI + 配置文件**（最常用）：写一个 `e2b.Dockerfile` 描述环境，再用 CLI 构建。

   ```bash
   npm install -g @e2b/cli      # 安装 CLI
   e2b auth login               # 登录
   e2b template init            # 生成 e2b.toml 和 e2b.Dockerfile
   # 编辑 e2b.Dockerfile，例如：
   #   FROM e2bdev/code-interpreter:latest
   #   RUN pip install pandas scikit-learn torch
   e2b template build           # 构建并上传，得到一个 template ID / 名字
   ```

2. **用构建好的模板创建沙箱**：把名字或 ID 传给 `create`。

   ```python
   sbx = Sandbox.create(template="my-ml-template")
   ```

> 模板进阶（基础镜像、start/ready 命令、缓存、版本标签）见官方 Template 文档，入门阶段会用默认模板即可。

## 十三、生产实践与避坑

- **务必回收沙箱**：优先 `with`；不能用 `with` 时把 `kill()` 放进 `finally`。忘关的沙箱会一直占配额、持续计费，直到超时。
- **超时要量体裁衣**：`timeout` 给够任务时间但别过长。长任务可在运行中 `set_timeout` 续命，或开启"超时即暂停"。
- **`run_code` 还是 `commands.run`**：要状态/富输出/做分析 → `run_code`；装包、起服务、用命令行工具 → `commands.run`。
- **错误要回灌而非吞掉**：把 `execution.error.traceback` 交给 LLM，它往往能自我修正——这是 agent 健壮性的关键。
- **用 metadata 做检索**：创建时打上 `{"user_id": ..., "session": ...}`，之后用 `Sandbox.list(query=...)` 精确找到/清理某用户的沙箱。
- **善用 persistence**：多轮对话、需要保留环境的长会话，用 `pause` + `connect` 而不是每次重建。
- **出网控制**：默认可出网；处理高敏场景可在创建时 `allow_internet_access=False` 切断外联。
- **并发与配额**：账户有并发沙箱数上限，批量任务前先确认配额，必要时做排队。

## 十四、API 速查表

```python
from e2b_code_interpreter import Sandbox

# ── 生命周期 ──────────────────────────────
sbx = Sandbox.create(timeout=300, metadata={...}, envs={...})
with Sandbox.create() as sbx: ...           # 自动回收（推荐）
sbx.sandbox_id                              # 沙箱 ID
sbx.set_timeout(600)                        # 续命（从现在重新计时）
sbx.get_info()                              # 元信息
sbx.kill()                                  # 销毁
Sandbox.connect(sandbox_id)                 # 凭 ID 连接（暂停态会自动恢复）
sbx.pause()                                 # 暂停（冻结全部状态）
Sandbox.list(query=SandboxQuery(...))       # 列出（返回分页器）

# ── 执行代码（有状态 Jupyter 内核）────────
ex = sbx.run_code(code, language="python")
ex.logs.stdout      # list[str]
ex.logs.stderr      # list[str]
ex.results          # list[Result]：.text/.png/.jpeg/.html/.svg/.chart
ex.text             # 最后一个结果的文本
ex.error            # None 或 .name/.value/.traceback
ctx = sbx.create_code_context(cwd="/home/user", language="python")

# ── 文件系统 ──────────────────────────────
sbx.files.write(path, data)                 # data: str/bytes/file
sbx.files.write([{"path":..,"data":..}, ..]) # 批量
sbx.files.read(path, format="text")         # "text"/"bytes"/"stream"
sbx.files.list(path)                        # list[EntryInfo]
sbx.files.exists(path) / .remove(path)
sbx.files.rename(old, new) / .make_dir(path)
sbx.files.watch_dir(path, recursive=False)

# ── 命令 / shell ──────────────────────────
r = sbx.commands.run("ls -l")               # CommandResult: .stdout/.stderr/.exit_code
sbx.commands.run(cmd, on_stdout=cb)         # 流式
h = sbx.commands.run(cmd, background=True)  # CommandHandle: .pid/.wait()/.kill()
```

### 延伸阅读（官方文档）

- 总入口：<https://e2b.dev/docs>
- 快速上手：<https://e2b.dev/docs/quickstart>
- 接入 LLM：<https://e2b.dev/docs/quickstart/connect-llms>
- 沙箱生命周期：<https://e2b.dev/docs/sandbox>
- 持久化（pause/resume）：<https://e2b.dev/docs/sandbox/persistence>
- 文件系统：<https://e2b.dev/docs/filesystem>
- 命令执行：<https://e2b.dev/docs/commands>
- 代码解释 / 图表：<https://e2b.dev/docs/code-interpreting/analyze-data-with-ai>
- 自定义模板：<https://e2b.dev/docs/template/quickstart>
- Python SDK 参考：<https://e2b.dev/docs/sdk-reference/python-sdk/v2.14.1/sandbox_sync>
- Cookbook（示例集）：<https://e2b.dev/docs/cookbook>